# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hussainhhgh/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
!git clone https://github.com/Hussainhhgh/flyrank-ml-internship.git 2>/dev/null


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1 — Feature Importance (Random Forest → Health Score), ML Appendix.**
The paper reports Average Position (43%), Impressions (32%), and Scroll Depth (15%) as the top predictors of health score, and is careful to note "health score is partly constructed from inputs such as position and impressions." My methodology question: since Health Score is explicitly defined as Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts), doesn't that make this closer to measuring the composite's own formula weights than discovering an independent pattern? The paper already flags this ("read this as model behavior, not... external causation"), which I think is the right caution — my question is whether a "feature importance" framing is the clearest way to communicate that, versus something like "formula sensitivity," since a reader skimming the bar chart could still walk away thinking position independently causes higher health.

**Finding 2 — Growth Prediction (Logistic Regression, 71% holdout accuracy), ML Appendix.**
The paper reports 71% holdout accuracy predicting growing vs. declining content, with content_age, days_since_update, and days_visible as the strongest signals. My methodology question: where does the label come from, and does the split design carry the claim? The paper's Trend Direction definition is calculated from 30d-vs-prev-30d impression change, which is directly time-based — but the methodology section only mentions an "80/20 split" without stating whether it's time-aware or grouped by content/brand. If the same brand's pages appear in both the train and test 80/20 split, the model could be partly learning brand-specific patterns rather than a generalizable growth signal, similar to the client-leakage risk I addressed in my own Week-5 model. This isn't a claim the paper is wrong — just a detail I'd want confirmed before fully trusting the 71% number as an out-of-sample estimate on genuinely new content.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Before/after honest split:** Re-running my Week-5 Random Forest under a naive row-level split (no client grouping) gives Precision@50 = 0.880 — a full 22 points higher than my honest client-grouped result of 0.660. This gap is the leakage risk made visible: with a row-level split, the same client's pages can appear in both train and test, so the model partly learns client-specific patterns (a particular client's content style, industry, reporting quirks) rather than a signal that generalizes to genuinely new clients. The client-grouped 0.660 is the number I trust and the one I reported in my Week-5 submission — it reflects performance on clients the model has never seen, which is the real-world deployment scenario. This 0.22 gap is itself a useful, concrete illustration of why validation design matters as much as model choice.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv('flyrank-ml-internship/data/raw/content_refresh_anonymized.csv')
df['target'] = (df['trend_direction'] == 'down').astype(int)
features = ['content_age_days', 'avg_position', 'ctr', 'impressions_90d',
            'engagement_rate', 'search_volume']
df_model = df.dropna(subset=features + ['target'])

def precision_at_k(y_true, scores, k=50):
    top_k_idx = pd.Series(scores).nlargest(k).index
    return y_true.iloc[top_k_idx].mean()

X = df_model[features]
y = df_model['target']
X_train_naive, X_test_naive, y_train_naive, y_test_naive = train_test_split(
    X, y, test_size=0.3, random_state=42)

rf_naive = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
rf_naive.fit(X_train_naive, y_train_naive)
naive_scores = rf_naive.predict_proba(X_test_naive)[:, 1]
naive_p50 = precision_at_k(y_test_naive.reset_index(drop=True), pd.Series(naive_scores))

clients = df_model['client_id'].unique()
train_clients, test_clients = train_test_split(clients, test_size=0.3, random_state=42)
train_df = df_model[df_model['client_id'].isin(train_clients)]
test_df = df_model[df_model['client_id'].isin(test_clients)]
X_train_grp, y_train_grp = train_df[features], train_df['target']
X_test_grp, y_test_grp = test_df[features], test_df['target']

rf_grp = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
rf_grp.fit(X_train_grp, y_train_grp)
grp_scores = rf_grp.predict_proba(X_test_grp)[:, 1]
grp_p50 = precision_at_k(y_test_grp.reset_index(drop=True), pd.Series(grp_scores))

print(f"BEFORE (naive row-level split): Precision@50 = {naive_p50:.3f}")
print(f"AFTER  (client-grouped split):  Precision@50 = {grp_p50:.3f}")
print(f"Gap: {naive_p50 - grp_p50:.3f}")

BEFORE (naive row-level split): Precision@50 = 0.880
AFTER  (client-grouped split):  Precision@50 = 0.660
Gap: 0.220


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

**Leakage audit (Week 3 hunt, repeated on final feature set):** Re-checking every feature in my Week-5 model for availability at decision time. All six features (content_age_days, avg_position, ctr, impressions_90d, engagement_rate, search_volume) are properties knowable before any future outcome is observed — none of them are derived from the label. trend_direction and trend_pct remain correctly excluded, since trend_direction is directly computed from trend_pct, which is the same signal used to construct my target variable. I also ran a correlation check between each feature and the target to confirm no feature is suspiciously close to a disguised copy of the label. The strongest correlation is content_age_days at -0.179 — weak, and far from the ±0.9+ range that would signal hidden leakage. No feature shows a suspiciously high relationship with the target, which supports that the model's predictive power comes from combining several weak signals rather than one feature secretly encoding the answer.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

feature_audit = {
    'content_age_days': 'Known at any point in time. SAFE.',
    'avg_position': 'Reflects recent ranking, knowable before outcome. SAFE (note: 0 = "no data", not rank zero).',
    'ctr': 'Reflects recent click behavior. SAFE.',
    'impressions_90d': 'Trailing 90-day window, available at decision time. SAFE.',
    'engagement_rate': 'Trailing engagement metric. SAFE.',
    'search_volume': 'External keyword metric, independent of performance history. SAFE.',
}
for feat, note in feature_audit.items():
    print(f"{feat}: {note}")

print("\nExcluded (label-derived, correctly NOT used as features):")
print("trend_direction — derived from trend_pct")
print("trend_pct — defines the target itself")

correlations = df_model[features + ['target']].corr()['target'].sort_values(ascending=False)
print("\nFeature correlation with target:")
print(correlations)

content_age_days: Known at any point in time. SAFE.
avg_position: Reflects recent ranking, knowable before outcome. SAFE (note: 0 = "no data", not rank zero).
ctr: Reflects recent click behavior. SAFE.
impressions_90d: Trailing 90-day window, available at decision time. SAFE.
engagement_rate: Trailing engagement metric. SAFE.
search_volume: External keyword metric, independent of performance history. SAFE.

Excluded (label-derived, correctly NOT used as features):
trend_direction — derived from trend_pct
trend_pct — defines the target itself

Feature correlation with target:
target              1.000000
search_volume      -0.019103
engagement_rate    -0.021607
impressions_90d    -0.032653
ctr                -0.041109
avg_position       -0.070770
content_age_days   -0.179154
Name: target, dtype: float64


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Claim rewrite:** My boldest earlier claim, from ML-08, was: "Random Forest reached 0.66 Precision@50 — an 18-point lift over the hand-written rule." Rewritten in safe language: "Under a client-grouped holdout split, the Random Forest model achieved an observed Precision@50 of 0.66 on this dataset, compared to 0.48 for the hand-written baseline rule — a measured, directional improvement of 0.18. This result is decision-support only: it reflects performance on this specific 30-client sample and should not be read as a guaranteed or causal improvement on unseen clients, new time periods, or different content types. For contrast, a naive row-level split (no client grouping) inflated the same model's score to 0.88 — a 0.22-point gap that shows how much validation design alone can distort a result. The correlation check in Section 3 also confirms no individual feature is strongly correlated with the target on its own (max 0.179), suggesting the model's lift over the baseline comes from combining several weak signals rather than any single dominant predictor."

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

baseline_p50 = 0.48
print(f"Baseline rule Precision@50: {baseline_p50:.2f}")
print(f"Random Forest (naive split): {naive_p50:.2f}")
print(f"Random Forest (client-grouped, honest): {grp_p50:.2f}")
print(f"Lift over baseline (honest split): {grp_p50 - baseline_p50:.2f}")
print(f"\nStrongest single feature correlation with target: {correlations.drop('target').abs().max():.3f}")
print("No single feature dominates — supports 'combined weak signals' framing in the claim rewrite.")


Baseline rule Precision@50: 0.48
Random Forest (naive split): 0.88
Random Forest (client-grouped, honest): 0.66
Lift over baseline (honest split): 0.18

Strongest single feature correlation with target: 0.179
No single feature dominates — supports 'combined weak signals' framing in the claim rewrite.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.